# 💻 How to Talk to an LLM with Code?
### A step-by-step guide to conversations, styles, memory, and tools

## 0️⃣ Setup ⚙️

Before starting, make sure your environment is ready.
You’ll need to install dependencies, configure your API keys, and check connectivity.

**More guidelines can be found in: README.md and connectivity_check notebook**


In [ ]:
import os
import json
import requests

from dotenv import dotenv_values
from openai import OpenAI

secrets = dotenv_values("../.env")
os.environ["OPENAI_API_KEY"] = secrets["OPENAI_API_KEY"]

## 1️⃣ Our First Conversation 💬

Let’s begin by starting a simple dialogue with the AI.
This will help you see how prompts and responses flow.

In [ ]:
client = OpenAI()
model = "gpt-4o"

message = "Write me an one-sentence bed-time story about an astronaut."

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[{
        "role": "user", "content": message
    }]
)

In [ ]:
response.choices[0].message.content

## 2️⃣ Defining Response Style with `SYSTEM_PROMPT` 🎭

Learn how to shape the AI’s personality and tone using the `system` role.
This is where you decide whether your assistant is a tutor, a poet, or even a pirate!

In [ ]:
client = OpenAI()
model = "gpt-4o"

system_prompt = "You are a dadaist poet. Respond with absurd, dreamlike, and surprising associations."
message = "Write me an one-sentence bed-time story about an astronaut."

In [ ]:
response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": message}
    ]
)

In [ ]:
response.choices[0].message.content

## 3️⃣ Assistant 🤖 — Memory and Chat in Action

The `assistant` role represents the AI’s responses.
Here you’ll explore how the model can remember context across turns and hold a real conversation, not just answer single questions.


In [ ]:
client = OpenAI()
model = "gpt-4o"

system_prompt = "You are a friendly tutor who teaches clearly with simple examples."
user_message1 = "Give me the current weather in Warsaw, Poland, in Celcius."

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_message1}
]

response = client.chat.completions.create(
    model=model,
    messages=messages
)

In [ ]:
text_content = response.choices[0].message.content
messages.append({"role": "assistant", "content": text_content})
print(text_content)

In [ ]:
user_message2 = "Check the weather in Warsaw right now using OpenWeatherMap and return temperature and description in Celsius."
messages.append({"role": "user", "content": user_message2})

response = client.chat.completions.create(
    model=model,
    messages=messages
)

In [ ]:
text_content = response.choices[0].message.content
print(text_content)

In [ ]:
user_message3 = ("Is there a mechanism in the Chat Completions API that allows the assistant to use external tools," +
                 "such as OpenWeatherMap? If so, what is it called?")
messages.append({"role": "user", "content": user_message3})

response = client.chat.completions.create(
    model=model,
    messages=messages
)

In [ ]:
text_content = response.choices[0].message.content
print(text_content)

## 4️⃣ Exploring the Function Calling Mechanism 🛠️

Learn how to extend the assistant with external tools.
This section shows how the model can call functions (like fetching real-time weather data) instead of only generating text.

In [ ]:
def get_weather(city, secrets):
    url = """https://api.openweathermap.org/data/2.5/weather?q={}&appid={}&units=metric""".format(city, secrets["OPEN_WEATHER_API_KEY"])
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception("An error has occured, code: {}, reason: {}".format(response.status_code, response.reason))
    data_dict = response.json()

    return json.dumps({
        "city": data_dict['name'],
        "weather": data_dict['weather'][0]['description'],
        "temperature": data_dict['main']['temp'],
        "unit": "Celsius"
    })

In [ ]:
weather_function_spec = {
    "name": "get_weather",
    "description": "Get the current temperature & weather for a city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city",
            },
        },
        "required": ["city"]
    }
}

In [ ]:
client = OpenAI()
model = "gpt-4o"

system_prompt = "You are a concise assistant who gives clear, direct answers without unnecessary detail."
user_message1 = "Give me the current weather in Warsaw, Poland, in Celcius."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_message1}
]

response = client.chat.completions.create(
    model="gpt-4",
    messages=messages,
    functions=[weather_function_spec]
)

message = response.choices[0].message
messages.append(message)
message

In [ ]:
args = json.loads(message.function_call.arguments) # json to dict
get_weather_result = get_weather(args["city"],secrets)

get_weather_result

In [ ]:
messages.append({"role": "function", "name": "get_weather", "content": get_weather_result})
response  = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    functions=[weather_function_spec]
)
response.choices[0].message.content

## 5️⃣ Practice: Test Your Knowledge 🧠✨

Now it’s time to put your newly gained knowledge and prompting skills into action.
Try these prompts and see how the model responses:

- **Prompt 1a:** *Where is the weather better right now, London or New York?* 🌍
- **Prompt 1b:** *Which city is colder, and by how many degrees?* ❄️
- **Prompt 2:** *Give me the current weather in the five biggest cities in Europe. Sort the cities by temperature.* 🏙️

### 5.1 ✍️ Exercise
*Where is the better weather right now, London or New York?* 🌍

In [ ]:
def get_weather(city, secrets):
    url = """https://api.openweathermap.org/data/2.5/weather?q={}&appid={}&units=metric""".format(city, secrets["OPEN_WEATHER_API_KEY"])
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception("An error has occured, code: {}, reason: {}".format(response.status_code, response.reason))
    data_dict = response.json()

    return json.dumps({
        "city": data_dict['name'],
        "weather": data_dict['weather'][0]['description'],
        "temperature": data_dict['main']['temp'],
        "unit": "Celsius"
    })

In [ ]:
weather_function_spec = {
    "name": "get_weather",
    "description": "Get the current temperature & weather for a city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city",
            },
        },
        "required": ["city"]
    }
}

In [ ]:
client = OpenAI()
model = "gpt-4o"

system_prompt = "You are a concise assistant who gives clear, direct answers without unnecessary detail."
user_message1 = "Where is the weather better right now, London or New York?"

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_message1}
]

response = client.chat.completions.create(
    model="gpt-4",
    messages=messages,
    functions=[weather_function_spec]
)

message = response.choices[0].message
messages.append(message)
message

In [ ]:
args = json.loads(message.function_call.arguments)
get_weather_result = get_weather(args["city"],secrets)

get_weather_result

In [ ]:
messages.append({"role": "function", "name": "get_weather", "content": get_weather_result})
response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    functions=[weather_function_spec]
)

message = response.choices[0].message
messages.append(message)
message

In [ ]:
args = json.loads(message.function_call.arguments)
get_weather_result = get_weather(args["city"],secrets)

get_weather_result

In [ ]:
messages.append({"role": "function", "name": "get_weather", "content": get_weather_result})
response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    functions=[weather_function_spec]
)

message = response.choices[0].message
messages.append(message)
message

*Which city is colder, and by how many degrees?* ❄️

In [ ]:
user_message2 = "Which city is colder, and by how many degrees?"

messages.append({"role": "user", "content": user_message2})
response = client.chat.completions.create(
    model="gpt-4o",
    messages=messages,
    functions=[weather_function_spec]
)

message = response.choices[0].message
messages.append(message)
message

### 📌 Conclusion

> - The more tool calls a query requires, the more code you need
> - — unless you use a **loop** to handle the repetition.

### 5.2 ✍️ Exercise
*Give me the current weather in the five biggest cities in Europe. Sort the cities by temperature.* 🏙️

In [ ]:
def get_weather(city, secrets):
    url = """https://api.openweathermap.org/data/2.5/weather?q={}&appid={}&units=metric""".format(city, secrets["OPEN_WEATHER_API_KEY"])
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception("An error has occured, code: {}, reason: {}".format(response.status_code, response.reason))
    data_dict = response.json()

    return json.dumps({
        "city": data_dict['name'],
        "weather": data_dict['weather'][0]['description'],
        "temperature": data_dict['main']['temp'],
        "unit": "Celsius"
    })

In [ ]:
weather_function_spec = {
    "name": "get_weather",
    "description": "Get the current temperature & weather for a city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city",
            },
        },
        "required": ["city"]
    }
}

In [ ]:
client = OpenAI()
model = "gpt-4o"

system_prompt = "You are a concise assistant who gives clear, direct answers without unnecessary detail."
user_message1 = "Give me the current weather in the five biggest cities in Europe. Sort the cities by temperature."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_message1}
]

In [ ]:
while True:
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        functions=[weather_function_spec]
    )

    message = response.choices[0].message
    messages.append(message)

    if message.function_call is not None:
        print(f"\n🔧 Tool call requested: {message.function_call.name} "
              f"with arguments {message.function_call.arguments}")

        if message.function_call.name == "get_weather":
            args = json.loads(message.function_call.arguments)
            weather_response = get_weather(args["city"], secrets)
            print(f"📦 Tool response: {weather_response}")

            messages.append({
                "role": "function",
                "name": "get_weather",
                "content": weather_response
            })

    elif response.choices[0].finish_reason == "stop":
        print(f"\n💬 Assistant: {message.content}")
        break

### 📌 Conclusion

The loop works, but remains quite limited.

> - Supports only **one tool** (`get_weather`).
> - Tied to a **single model choice** (OpenAI `gpt-4o`).
> - No proper **error handling or retries** (timeouts, 4xx/5xx).
> - …and these are just a few of its limitations.


## 6️⃣ Conclusions & Summary 📝

In Part 1, we explored the foundations of *talking to an LLM with code*. Step by step, we learned:

- 💬 **System Prompting** — how to shape the assistant’s style and behavior.
- 🗂️ **Conversation & Memory** — moving beyond Q&A into multi-turn chat.
- 🛠️ **Function Calling** — defining and wiring tools (like `get_weather`) into the model.

From these lessons, two key insights stand out:

- 🔄 **Need for a Loop** — conversations and tool calls often require multiple iterations of reasoning, which must be managed in code.
- 🧩 **Need for Abstraction** — building everything manually (tools, loops, storage, etc.) quickly becomes fragile and hard to maintain.

➡️ These lessons naturally lead us to the next topics:
- **Agents** 🤖 — giving models the ability to reason, plan, and act.
- **Frameworks** 🏗️ — providing structure so we don’t drown in boilerplate code.

👉 These will be the focus of **Part 2 (Agents)** and **Part 3 (Frameworks)**.
